# REQUIREMENTS

CREATE A 🔑 SECRET NAMED NGROK_KEY

https://dashboard.ngrok.com/

#INSTRUCTIONS
Use T4 GPU on runtime type and latest runtime version

Run All and wait for NGROK LINK

In [ ]:
# @title Requirement Check
# @markdown Allow Access for gdrive and secrets
from google.colab import drive, userdata, auth
from googleapiclient.discovery import build
from IPython.display import clear_output
import os
import sys
#DRIVE SETUP
drive.mount('/content/drive')

SHORTCUT_NAME = "BA-CHATBOT"
SHORTCUT_TARGET_PATH = f"/content/drive/MyDrive/{SHORTCUT_NAME}"
PUBLIC_FOLDER_ID = "1lKNUGjIwZhKwu_htckR8hAsjh6S-SMUj"

if not os.path.exists(SHORTCUT_TARGET_PATH):
    print(f"'{SHORTCUT_NAME}' Shortcut not detected in My Drive. Initiating automatic link creation...")
    auth.authenticate_user()
    try:
        drive_service = build('drive', 'v3')
        shortcut_metadata = {
            'name': SHORTCUT_NAME,
            'mimeType': 'application/vnd.google-apps.shortcut',
            'shortcutDetails': {
                'targetId': PUBLIC_FOLDER_ID
            }
        }
        file = drive_service.files().create(body=shortcut_metadata, fields='id').execute()
        drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print(f"Failed to create shortcut via API: {e}")
else:
    print(f"Shortcut verified")

#NGROK CHECK
NGROK_KEY = userdata.get('NGROK_KEY')
if not NGROK_KEY or not str(NGROK_KEY).strip():
  print("NGROK_KEY NOT FOUND")
  sys.exit("Execution stopped: NGROK_KEY is required.")
else:
  print("NGROK_KEY is found")

Mounted at /content/drive
Shortcut verified
NGROK_KEY is found


In [ ]:
# @title INITIAL SETUP
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip install flask flask-cors requests pillow pyngrok
!git clone https://github.com/modelscope/DiffSynth-Studio.git
%cd DiffSynth-Studio
!pip install -e .
!pip install modelscope torch torchvision transformers accelerate rapidfuzz

clear_output(wait=True)
print("FINISHED INSTALLING")

FINISHED INSTALLING


In [ ]:
# @title  MODEL SETUP
import torch
import random
from diffsynth.pipelines.anima_image import AnimaImagePipeline, ModelConfig
from llama_cpp import Llama

filePaths = "/content/drive/MyDrive/BA-CHATBOT/Anima"
loraPaths = f"{filePaths}/Loras"
MODEL_PATH = f"{filePaths}/llama 8b.gguf"

vram_config = {
    "offload_dtype": torch.float16,
    "offload_device": "cuda",
    "onload_dtype": torch.float16,
    "onload_device": "cuda",
    "preparing_dtype": torch.float16,
    "preparing_device": "cuda",
    "computation_dtype": torch.float16,
    "computation_device": "cuda",
}

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=8192,
    n_gpu_layers=-1,
    chat_format="llama-3"
)

pipe = AnimaImagePipeline.from_pretrained(
    torch_dtype=torch.float16,
    device="cuda",
    model_configs=[
        ModelConfig(path=f"{filePaths}/anima-base-v1.0.safetensors", **vram_config),
        ModelConfig(path=f"{filePaths}/qwen_3_06b_base.safetensors", **vram_config),
        ModelConfig(path=f"{filePaths}/qwen_image_vae.safetensors", **vram_config),
    ],
    tokenizer_config=ModelConfig(path=f"{filePaths}/Qwen/"),
    tokenizer_t5xxl_config=ModelConfig(path=f"{filePaths}/StableDiffusion/"),
    vram_limit=torch.cuda.mem_get_info()[1] / (1024 ** 3) - 0.5,
)

clear_output(wait=True)
print("MODELS LOADED")

MODELS LOADED


In [ ]:
#@title FUNCTIONS AND CONFIGURATIONS
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
import json
import time
import base64
from io import BytesIO
from pyngrok import ngrok
from rapidfuzz import process, fuzz
import re

#PROMPTS
SYSTEM_PROMPT_FILE = f"{filePaths}/TaggerPrompt.txt"
with open(SYSTEM_PROMPT_FILE, "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()


#TAGDATABASE
tagPromptPath = f"{filePaths}/TaggerLLM"
with open(f"{tagPromptPath}/general_tags.json", "r", encoding="utf-8") as f:
    tagData = json.load(f)
tagDB = {}
for node in tagData.values():
    tagDB[node["title"]] = {
        "body": node["body"],
        "links": node["is_linking_to"],
        "category" : node.get("general_category", "action")
    }
with open(f"{tagPromptPath}/tags_EntryPoint.json", "r", encoding="utf-8") as f:
    entryTags = json.load(f)
entryTags = {"active_points" : entryTags}
tag_list = tagDB.keys()



NGROK_KEY = userdata.get('NGROK_KEY')
ngrok.set_auth_token(NGROK_KEY)

#CHARACTERS
character = ["Kasumi", "Natsu", "Kei"]
current_character = character[0]

character_tags = {
    "Kasumi" : {
        "tag" : "kinugawa_kasumi",
        "outfit" : {
            "default" : ["white coat", "red shirt", "black shorts"],
            "swimsuit" : ["striped bikini, sunglasses, swimsuit, twintails, eyewear on head"]
            }
    },
    "Natsu" : {
        "tag" : "Yutori_natsu",
        "outfit" : {
            "default": ["school uniform", "blue skirt"],
            "band" : ["white shirt", "yellow jacket"]}
    },
    "Kei" : {
        "tag" : "tendou_kei",
        "outfit" : {
            "default": ["school uniform", "blue skirt"]
            }
    },
    "Hina" : {
        "tag" : "sorasaki_hina",
        "outfit" : {
            "default" : ["coat, black skirt, black gloves"],
            "swimsuit" : ["school swimsuit"],
            "dress" : ["elbow gloves, necklace, pendant, purple dress, purple gloves"]
            }
    },
    "Seia" : {
        "tag" : "yurizono_seia",
        "outfit" : {
            "default" : ["fox ears, white dress, sleeveless, white pantyhose"],
            "swimsuit" : ["fox ears, swimsuit, visor cap, yellow jacket"]
            }
    },
    "Hoshino" : {
        "tag" : "takanashi_hoshino",
        "outfit" : {
            "default" : ["necktie, black skirt, collared shirt"],
            "swimsuit" : ["frils, jacket, swimsuit, blue-tinted eyewear, eyewear on head"]
        }
    },
    "Ichika" : {
        "tag" : "nakamasa_ichika",
        "outfit" : {
            "default" : ["black choker, black serafuku, black sailor collar, red armband, closed eyes"],
            "swimsuit" : ["closed eyes, sunglasses, eyewear on head, bikini, black shorts, black jacket, red belt"]
        }
    },
    "Koharu" : {
        "tag" : "shimoe_koharu",
        "outfit" : {
            "default" : ["black hat, black shirt, white sailor collar, school uniform, skirt, long sleeves"],
            "magical girl" : ["black bow, black bikini, floral print bikini, swimsuit"]
        }
    },
    "Hifumi" : {
        "tag" : "ajitani_hifumi",
        "outfit" : {
            "default" : ["white cardigan, sailor collar, black pantyhose, pleated skirt, school uniform"],
            "magical girl" : ["ribbon-trimmed bikini, white bikini, swimsuit, eyewear on head, sunglasses, frills"]
        }
    },
    "Azusa" : {
        "tag" : "shirasu_azusa",
        "outfit" : {
            "default" : ["wings, long sleeves, sailor collar. black dress, frills, jacket, white jacket, school uniform"],
            "swimsuit" : ["wings, bikini, bikini skirt, swimsuit, swimsuit, frills, necklace, strapless"]
        }
    },
    "Hanako" : {
        "tag" : "urawa_hanako",
        "outfit" : {
            "default" : ["white shirt, sailor collar, pink bow, blue sailor collar, school uniform, skirt"],
            "swimsuit" : ["bottomless, swimsuit, see-through shirt, wet shirt, red bikini"]
        }
    },
    "Reisa" : {
        "tag" : "uzawa_reisa",
        "outfit" : {
            "default" : ["low twintails, pleated skirt, grey skirt, grey shirt, black jacket, school uniform, long sleeves, sailor collar"],
            "magical girl" : ["wing hair ornament, drill hair, puffy sleeves, short sleeves, magical girl, striped pantyhose, purple bow, dress"]
        }
    },
    "Suzumi" : {
        "tag" : "morizuki_suzumi",
        "outfit" : {
            "default" : ["long sleeves, school uniform, grey skirt, frilled skirt, black thighhighs, red ribbon, grey jacket"],
            "magical girl" : ["magical girl, pink skirt, bow, pink dress, detached sleeves, crescent hair ornament, white thighhighs"]
        }
    },
    "Hare" : {
        "tag" : "omagari_hare",
        "outfit" : {
            "default" : ["white coat, black hoodie, hood down, cross hair ornament"],
            "camp" : ["black scarf, black pantyhose, green jacket, white hat, grey shirt"],
            "pajamas" : ["pajamas, bottomless, hood down, cross hair ornament, black shirt, oversized shirt"]
        }
    },
    "Kotama" : {
        "tag" : "otose_kotama",
        "outfit" : {
            "default" : ["black-framed eyewear, black gloves, black jacket, blue necktie, headphones, white shirt, black choker, white skirt"],
            "camp" : ["black-framed eyewear, yellow jacket, earmuffs, scarf, white gloves, black skirt, ponytail, black pantyhose"],
            "pajamas" : ["black-framed eyewear, purple shirt, purple pajamas, mask, mask on head, sleep mask, bottomless"]
        }
    },
    "Maki" : {
        "tag" : "konuri_maki",
        "outfit" : {
            "default" : ["hair bun, blue necktie, blue jacket, school uniform, white skirt"],
            "camp" : ["twin braids, brown jacket, red shirt, hat, black pantyhose, blue shorts"],
            "pajamas" : ["white shirt, sleeveless shirt, hair downm long hair, black shorts, pajamas"]
        }
    },
    "Chihiro" : {
        "tag" : "kagami_chihiro",
        "outfit" : {
            "default" : ["glasses, two-tone jacket, rabbit hair ornament, blue necktie, collared shirt, black skirt, school uniform"],
            "camp" : ["glasses, bucket hat, black gloves, black leggings, blue coat"],
            "pajamas" : ["glasses, black hairband, dolphin shorts, blue shorts, white sweater, pajamas"]
        }
    },
    "Shigure" : {
        "tag" : "mayoi_shigure",
        "outfit" : {
            "default" : ["weasel ears, black hat, grey jacket, grey gloves"],
            "hot spring" : ["weasel ears, bath yukata, blue sash, blue haori, yagasuri"]
        }
    },
    "Nodoka" : {
        "tag" : "amami_nodoka",
        "outfit" : {
            "default" : ["jacket, grey skirt, grey gloves, hat, grey coat, black pantyhose"],
            "hot spring" : ["vertical-striped kimono, white sash"]
        }
    },
    "Miyu" : {
        "tag" : "kasumizawa_miyu",
        "outfit" : {
            "default" : ["leaf on head, rabbit ears, green neckerchief, blue serafuku, blue skirt, white sailor collar, white pantyhose, school uniform"],
            "swimsuit" : ["twin braids, print bikini, straw hat, swimsuit"]
        }
    },
    "Miyako" : {
        "tag" : "tsukiyuki_miyako",
        "outfit" : {
            "default" : ["school uniform, rabbit ears, bulletproof vest, blue shirt, blue skirt, white sailor collar, hair down"],
            "swimsuit" : ["frills, blue one-piece swimsuit"]
        }
    },
    "Moe" : {
        "tag" : "kazekura_moe",
        "outfit" : {
            "default" : ["blue sweater, blue skirt, rabbit ears, black pantyhose, round eyewear"],
            "swimsuit" : ["rabbit ears, blue hoodie, cropped hoodie, black bikini, bikini bottom only, round eyewear"]
        }
    },
    "Saki" : {
        "tag" : "sorai_saki",
        "outfit" : {
            "default" : ["white helmet, blue shirt, belt pouch, blue serafuku, blue sailor collar, school uniform"],
            "swimsuit" : ["bucket hat, blue bikini, cropped jacket, swimsuit"]
        }
    },
    "Izuna" : {
        "tag" : "kuda_izuna",
        "outfit" : {
            "default" : ["fox ears, white shirt, pink neckerchief, pink scarf, floral print kimono, school uniform"],
            "dress" : ["fox ears, pink dress, pink choker, fur capelet"],
            "swimsuit" : ["fox ears, swimsuit, visor cap, striped bikini, denim shorts, pink scarf"]
        }
    },
    "Michiru" : {
        "tag" : "chidori_michiru",
        "outfit" : {
            "default" : ["raccoon ears, white shirt, school uniform, pink neckerchief, blue skirt. black scarf, black pantyhose"],
            "dress" : ["raccoon ears, white dress, elbow gloves, purple hat, white pantyhose"]
        }
    },
    "Tsukuyo" : {
        "tag" : "oono_tsukuyo",
        "outfit" : {
            "default" : ["lop rabbit ears, red neckerchief, purple scarf, school uniform, blue skirt, purple cardigan"],
            "dress" : ["lop rabbit ears, see-through cleavage, kimono dress"]
        }
    }

}

#STYLES
style_path = f"{loraPaths}/style/"
style_config = {
    "DEFAULT" : {
        "safetensor" : "Default.safetensors",
        "trigger" : "@BlueArchStyle"
    },
    "CLEAR" : {
        "safetensor" : "Clear - Vibestyle.safetensors",
        "trigger" : ""
    },
    "PASTEL" : {
        "safetensor" : "Pastel - VibeStyle.safetensors",
        "trigger" : ""
    },
    "CHIBI" : {
        "safetensor" : "Chibi - VibeStyle.safetensors",
        "trigger" : ""
    },
    "BISHOUJO" : {
        "safetensor" : "Bishoujo - VibeStyle.safetensors",
        "trigger" : ""
    },
    "FLAT COLOR" : {
        "safetensor" : "flat_color.safetensors",
        "trigger" : "flat color, no lineart"
    },
    "RL" : {
        "safetensor" : "RL.safetensors",
        "trigger" : ""
    }
 }

current_style = style_config["DEFAULT"]

def change_lora(selected_char, selected_style):
  pipe.clear_lora()
  turbo_lora = ModelConfig(path=f"{loraPaths}/anima-turbo-lora-v0.1.safetensors")
  pipe.load_lora(pipe.dit, turbo_lora, alpha=1)
  character_lora = ModelConfig(path=f"{loraPaths}/{selected_char}V1.safetensors")
  pipe.load_lora(pipe.dit, character_lora, alpha=1.0)
  style_lora = ModelConfig(path=f"{style_path}{selected_style['safetensor']}")
  pipe.load_lora(pipe.dit, style_lora, alpha=1.0)

change_lora(current_character, current_style)

def generate_image(prompt):
    negative_prompt = "worst quality, low quality, monochrome, zombie, interlocked fingers, Aissist, cleavage, logo, text, speech bubble, chat"

    seed = random.randint(1, 10_000_000)

    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        cfg_scale=1.0,
        seed=seed,
        num_inference_steps=20,
        height=512,
        width=512
    )
    return image

def find_matching_tags(tag, TAG_LIST):
  direct_match = False
  match = process.extract(tag, TAG_LIST, scorer=fuzz.WRatio, limit=10)
  new_tag_list = []
  for item in match:
    if item[1] == 100:
      new_tag_list = item[0]
      direct_match = True
      break
    elif item[1] >= 80:
      new_tag_list.append(item[0])

  return new_tag_list, direct_match

clear_output(wait=True)
print("CONFIGURATIONS READY")

CONFIGURATIONS READY


In [ ]:
#@title TAG BUILDER
def build_danbooru_prompt(chat, character_name, current_scene, player_name="Sensei"):
    lines = []

    for msg in chat[-7:-2]:
        role = msg.get("role")
        content = msg.get("content", "").strip()
        if not content:
            continue

        if role == "user":
            lines.append(f"{player_name}: {content}")
        else:
            lines.append(f"{character_name}: {content}")

    prompt = f"""
    [CONVERSATION HISTORY]
    {"\n".join(lines)}

    USE THIS AS HISTORY, JUST SAY 'SKIPPED' FOR NOW
    """
    recent_npc_msg = chat[-1].get("content", "").strip()
    recent_sensei_msg = ""
    if len(chat) >= 2:
      recent_sensei_msg = chat[-2].get("content", "").strip()


    recent_prompt = f"""
    [LAST SCENE]
    {str(current_scene)}
    [CURRENT CHAT]
    {player_name} : {recent_sensei_msg}
    {character_name} : {recent_npc_msg}
    """

    #LLAMA TAGPROMPT
    history_list = [
        {"role" : "system", "content" : SYSTEM_PROMPT},
        {"role" : "user", "content" : prompt},
        {"role" : "assistant", "content" : "SKIPPED"},
        {"role" : "user", "content" : recent_prompt}
    ]
    #LLAMA
    response = llm.create_chat_completion(
        messages=history_list,
        temperature=0.2,
        top_p=0.95,
        max_tokens=2000,
        repeat_penalty=1.05,
        response_format={
            "type": "json_object"
        }
    )

    tags = response["choices"][0]["message"]["content"]
    data_dict = json.loads(tags)
    current_scene = data_dict.get("scene", {})


    history_list.append({"role" : "assistant", "content" : tags})

    #--------------- LLAMA TAG EVALUATOR --------------------
    tag_evaluator_data = {
        "final_tags" : [],
        "natural_prompt" : data_dict.get("natural_prompt"),
        "invalid_tags" : [],
        "valid_tags" : []
    }

    full_tags_created = data_dict.get("tags")
    scenes = data_dict.get("scene")
    for value in scenes.values():
      if not value:
        continue

      if isinstance(value, str):
        value = re.sub(r"\s*,\s*", ",", value)
        value = value.split(",")
      full_tags_created = list(set(value + full_tags_created))

    for tag in full_tags_created:
      tag = tag.replace(" ", "_")
      tags, direct_match = find_matching_tags(tag, tag_list)
      if direct_match:
        tag_evaluator_data["final_tags"].append(tags)
      else:
        tag_evaluator_data["invalid_tags"].append(tag)
        tag_evaluator_data["valid_tags"] = list(set(tags + tag_evaluator_data["valid_tags"]))

    tag_evaluator_prompt = """
    You are a tag evaluator for danbooru tags. Your goal is to check the given tags for evaluation and decide by picking valid tags based on the tag list provided for each tags to be evaluated.

    Once the tags are evaluated, you need to create a new "natural_prompt" from the given "natural_prompt" based on **NATURAL PROMPT REWORK INSTRUCTIONS**
    ---
    **USER INPUT FORMAT**
    {"final_tags" : [],
    "natural_prompt" : "",
    "invalid_tags" : [],
    "valid_tags" : []
    ---
    **Main Rules**
    - Do not edit or get tags from the "final_tags", use purely as reference for already existing tags.
    - You need to evaluate the "invalid_tags" and find a replacement the "valid_tags" IF IT MAKES SENSE.
    - YOU CAN SKIP THE INVALID TAG ENTIRELY IF THERE ARE NO REPLACEMENTS FROM VALID TAGS
    - DO NOT ADD THE VALID TAG IN SELECTED TAGS IF IT DOES NOT MEET THE CRITERIAS
    - VALID TAGS ARE STRICTLY VALID TAGS YOU CAN USE, IT DOES NOT MEAN THEY ARE THE FINAL TAGS THAT YOU WILL ACTUALLY USE.

    ---
    ##SELECTION CRITERIA
    - STRICTLY SELECT TAGS FROM VALID TAGS
    - If there is a direct identical tag from invalid_tags on valid_tags, add the valid_tag in "selected_tags"
    - If there is a general tag about the invalid_tag in valid_tags, add the valid_tag in "selected_tags"

    ##REJECTION CRITERIA
    - STRICTLY REJECT THE VALID_TAG IF ITS UNRELATED TO THE INVALID_TAGS
    - DO NOT SELECT ANY TAGS FROM THE INVALID_TAGS
    - REJECTED TAGS WILL BE COMPLETELY SKIPPED AND CAN BE USED TO DESCRIBE THE NEW "natural_prompt"

    ---
    **NATURAL PROMPT INSTRUCTIONS**
    - Improve the given "natural_prompt" by adding additional details based on "invalid_tags"
    - If an invalid tag is skipped then describe how it is used on the scene. Completely remove the skipped invalid tag right after.

    ---
    **OUTPUT_INSTRUCTIONS**
    - STRICTLY OUTPUT ON THIS JSON FORMAT.
    - DO NOT OUTPUT ANY TEXTS, REASONING OR OTHER RESPONSES OTHER THAN THE JSON OUTPUT.
    - DO NOT ADD TAGS UNRELATED TO INVALID_TAGS IN SELECTED_TAGS OR THE INVALID_TAGS THEMSELVES.
    {
    "selected_tags" : ["tag_1", "tag_2", "tag_3"],
    "natural_prompt" : "reworked_natural_prompt"
    }

    - IF THE INVALID TAGS OR VALID TAGS ARE EMPTY, RESPOND WITH EMPTY "selected_tags" AND THE IMPROVED "natural_prompt"

    **USER INPUT**

    """
    tag_evaluator_prompt += str(tag_evaluator_data)

    history_list.append({"role" : "user", "content": tag_evaluator_prompt})

    evaluator_response = llm.create_chat_completion(
        messages=history_list,
        temperature=0.2,
        top_p=0.95,
        max_tokens=2000,
        repeat_penalty=1.05,
        response_format={
            "type": "json_object"
        }
    )

    evaluator_message = evaluator_response["choices"][0]["message"]["content"]
    evaluator_data = json.loads(evaluator_message)
    final_tags = list(set(tag_evaluator_data["final_tags"] + evaluator_data.get("selected_tags")))
    final_tags = [tag.replace("_", " ") for tag in final_tags]


    #-------------NATURAL PROMPT CLAENER-----------------
    cleaner_prompt = f"""
    You are a natural prompt cleaner. Your job is to rewrite the given natural prompt to be extremely short and concise, following strict rules.

    RULES:
    1. The character is '{character_name}'. Always refer to her as 'she'.
    2. Refer to Sensei as 'the viewer'. Do not use names or first person.
    3. Output ONLY ONE sentence, maximum 15 words.
    4. Describe ONLY ONE visible action. No sequences (remove 'then', 'before', 'after', 'first', 'next').
    5. Remove all internal thoughts, feelings, and dialogue (e.g., 'she feels', 'she thinks').
    6. Remove any descriptions of aftermath or completed actions (e.g., 'has already', 'had placed').
    7. If the prompt contains multiple actions, pick ONLY the most memorable action that is directed toward the viewer (handing, looking, reaching, speaking – without quoting words, or the viewer touching her).
    8. Keep only what can be seen in a single image (pose, expression, hand motion, gaze).
    9. Do NOT add any new content not present in the original prompt.
    10. Output ONLY the cleaned natural prompt text. No extra words, no quotes, no labels.

    PREVIOUS NATURAL_PROMPT:
    {evaluator_data.get("natural_prompt")}
    """

    history_list[-1] = ({"role" : "user", "content": cleaner_prompt})

    cleaned_response = llm.create_chat_completion(
        messages=history_list,
        temperature=0.2,
        top_p=0.95,
        max_tokens=2000,
        repeat_penalty=1.05
    )

    clean_natural_prompt = cleaned_response["choices"][0]["message"]["content"]

    for k, v in current_scene.items():
        if v is not None and v != "":
            last_scene[k] = v

    current_tags = final_tags
    meta_tags = ["1girl", "solo focus", "pov"]
    for tag in meta_tags:
      if not tag in current_tags:
        current_tags.append(tag)

    #LAST SCENE UPDATER
    new_last_scene = {}
    for tag in current_tags:
      db_tag = tag.replace(" ", "_")
      if db_tag in tagDB and tag not in meta_tags:
        current_category = tagDB[db_tag]["category"]
        if current_category not in new_last_scene:
          new_last_scene[current_category] = []
        new_last_scene[current_category].append(tag)

    for key, value in new_last_scene.items():
      last_scene[key] = ", ".join(value)


    return clean_natural_prompt, current_tags, tags

In [ ]:
#@title SERVER SETUP
#@markdown #RUN AND WAIT FOR THE NGROK URL
last_scene = {
    "subject" : "1girl, pov, solo focus",
    "location": "",
    "action": "",
    "mood": "",
    "time": "",
    "outfit": ", ".join(character_tags[current_character]["outfit"]),
    "objects": ""
}
chat_history = [

]

request_active = False


app = Flask(__name__)
CORS(app)
@app.route("/characters", methods=["POST"])
def get_characters():

    result = {}

    for name, data in character_tags.items():
        result[name] = {
            "outfit": data["outfit"]
        }

    return jsonify(result)
@app.route("/character/select", methods=["POST"])
def select_character():
    global current_character
    global last_scene
    global chat_history
    global current_style
    global request_active

    if request_active:
      return jsonify({
        "status": "busy"
      }), 429
    request_active = True

    try:
      data = request.json

      selected_character = data.get("character")
      selected_outfit = data.get("outfit")
      selected_style = data.get("style")

      chat_history = []
      current_history = data.get("chat", [])

      for i in current_history:
        i.pop("index", None)
        chat_history.append(i)

      if selected_character not in character_tags:
          return jsonify({
              "status": "error",
              "message": "Invalid character"
          }), 400

      current_character = selected_character

      # Reset scene memory
      last_scene = {
          "subject": "1girl, pov, solo focus",
          "location": "",
          "action": "",
          "mood": "",
          "time": "",
          "outfit": ", ".join(character_tags[current_character]["outfit"][selected_outfit]),
          "objects": ""
      }
      print(f"CHARACTER CHANGED \nCurrent Character: {current_character}")
      current_style = style_config[selected_style]
      change_lora(current_character, current_style)


      return jsonify({
          "status": "ok",
          "character": current_character,
          "scene": last_scene
      })
    except Exception as e:
      print("ERROR:", e)
      return jsonify({
          "status": "error",
          "message": str(e)
      }), 500
    finally:
      request_active = False



@app.route("/generate", methods=["POST"])
def generate():
    global last_scene
    global chat_history
    global current_style
    global request_active


    if request_active:
      return jsonify({
        "status": "busy"
      }), 429
    request_active = True

    try:
      data = request.json
      incoming_scene = data.get("scene", {})
      incoming_history = data.get("chat", [])

      current_index = 0

      for i in incoming_history:
        i.pop("index", None)
        if i not in chat_history:
          if len(incoming_history) <= 2:
            chat_history.append(i)
          else:
            chat_history.insert(current_index,i)
            current_index += 1
        else:
          current_index = chat_history.index(i) + 1


      for key, value in incoming_scene.items():
        last_scene[key] = value


      # ---------------- BUILD PROMPT ----------------
      print("Waiting for llama tags...")
      clean_natural_prompt, current_tags, tags = build_danbooru_prompt(chat_history, current_character, last_scene)



      style_trigger = current_style.get("trigger", "")
      tag_prompt = f'{character_tags[current_character]["tag"]}, {style_trigger}, {", ".join(current_tags)} \n{clean_natural_prompt}'

      print("\n================ TAGS ================\n")
      print(tag_prompt)
      print("\n=============================================\n")

      # ---------------- IMAGE GENERATION ----------------
      image = generate_image(tag_prompt)
      buffer = BytesIO()
      image.save(buffer, format="PNG")
      buffer.seek(0)
      encoded_image = base64.b64encode(buffer.read()).decode()

      # ---------------- RETURN TO TAMPERMONKEY ----------------
      return jsonify({
          "status": "ok",
          "tags": tags,
          "image": encoded_image,
          "scene": last_scene
      })
    except Exception as e:
      print("ERROR:", e)
      return jsonify({
          "status": "error",
          "message": str(e)
      }), 500
    finally:
      request_active = False
def run_app():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_app, daemon=True).start()
public_url = ngrok.connect(5000)
print("COPY THIS LINK:\n", public_url.public_url)


while True:
    time.sleep(60)